# AAMem Colab Pull And Run

This notebook pulls the latest code from `https://github.com/lenhokhoa23/j4f.git` and runs a fast smoke experiment. Change `MODEL_PRESET` to a small HF model when you have a GPU runtime.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = 'https://github.com/lenhokhoa23/j4f.git'
PROJECT_DIR = Path('/content/j4f') if Path('/content').exists() else Path.cwd().resolve()

if (PROJECT_DIR / '.git').exists():
    subprocess.run(['git', 'pull', '--ff-only'], cwd=PROJECT_DIR, check=True)
elif not PROJECT_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(PROJECT_DIR)], check=True)
else:
    print(f'{PROJECT_DIR} exists but is not a git repo; using it without pulling.')

os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
print('PROJECT_DIR =', PROJECT_DIR)


## Optional install for small local models

Run this only for `qwen2_5_1_5b`, `qwen2_5_3b`, `qwen2_5_7b`, `llama3_2_1b`, `llama3_2_3b`, or `gemma2_2b`.

In [ ]:
# Uncomment for HF local model runs on Colab GPU.
# %pip install -q transformers accelerate


In [ ]:
MODEL_PRESET = 'smoke'
# Examples: 'qwen2_5_1_5b', 'qwen2_5_3b', 'qwen2_5_7b', 'openai_api'

subprocess.run([
    sys.executable, '-m', 'aamem_lab.phase3_stale_runner',
    '--limit', '12',
    '--k', '5',
    '--dimensions', 'SR,PR,IPA',
    '--model-preset', MODEL_PRESET,
    '--include-prompts',
    '--out-dir', 'runs/colab_phase3_stale',
    '--tag', f'stale_n12_k5_{MODEL_PRESET}',
], cwd=PROJECT_DIR, check=True)


In [ ]:
subprocess.run([
    sys.executable, '-m', 'aamem_lab.phase2_answer_runner',
    '--dataset', 'actmem',
    '--limit', '10',
    '--k', '5',
    '--model-preset', MODEL_PRESET,
    '--include-prompts',
    '--out-dir', 'runs/colab_phase2_answer',
    '--tag', f'actmem_n10_k5_{MODEL_PRESET}',
], cwd=PROJECT_DIR, check=True)
